In [1]:
import pandas as pd
import os
from multiprocessing import Pool, cpu_count
from tqdm import tqdm

In [2]:
# ===== 配置 =====
INPUT_DIR = "../RawData"
OUTPUT_FILE = "ST12000NM0008_all.csv"
TARGET_MODEL = "ST12000NM0008"
N_WORKERS = max(1, cpu_count() - 1)


# ===== 单文件处理函数 =====
def process_file(file_path):
    try:
        df = pd.read_csv(file_path)

        if 'model' not in df.columns:
            return None

        filtered = df[df['model'] == TARGET_MODEL]

        return filtered if not filtered.empty else None

    except Exception as e:
        print(f"❌ 处理失败: {file_path}, 错误: {e}")
        return None


# ===== 主流程 =====
def main():
    files = [
        os.path.join(INPUT_DIR, f)
        for f in os.listdir(INPUT_DIR)
        if f.endswith(".csv")
    ]

    print(f"共发现 {len(files)} 个文件，开始处理...")

    results = []

    # ⭐ 核心：imap_unordered + tqdm
    with Pool(N_WORKERS) as pool:
        for res in tqdm(pool.imap_unordered(process_file, files), total=len(files)):
            if res is not None:
                results.append(res)

    if len(results) == 0:
        print("⚠️ 没有找到符合条件的数据")
        return

    final_df = pd.concat(results, ignore_index=True)
    # final_df.to_csv(OUTPUT_FILE, index=False)
    final_df.to_parquet("ST12000NM0008_all.parquet", index=False)

    print(f"✅ 完成！共筛选出 {len(final_df)} 行数据")
    print(f"📁 输出文件: {OUTPUT_FILE}")


if __name__ == "__main__":
    main()

共发现 2192 个文件，开始处理...


  2%|▏         | 34/2192 [00:04<02:54, 12.39it/s]/tmp/ipykernel_3698463/434190931.py:11: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
/tmp/ipykernel_3698463/434190931.py:11: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
  2%|▏         | 36/2192 [00:04<02:45, 13.05it/s]/tmp/ipykernel_3698463/434190931.py:11: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
  2%|▏         | 40/2192 [00:04<02:46, 12.91it/s]/tmp/ipykernel_3698463/434190931.py:11: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
  2%|▏         | 42/2192 [00:04<02:45, 12.98it/s]/tmp/ipykernel_3698463/434190931.py:11: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set lo

✅ 完成！共筛选出 41026987 行数据
📁 输出文件: ST12000NM0008_all.csv


In [3]:
df = pd.read_parquet("ST12000NM0008_all.parquet")

In [4]:
len(df)

41026987

In [5]:
sum(df["failure"] == 1)

2328